# 6.1 全参数微调 (Full Fine-Tuning)

全参数微调是微调的最基本形式，解冻模型所有参数在目标任务数据上进行训练。

**目的**：最大化任务适配效果

**基本原理**：解冻模型所有参数，在目标任务数据上进行训练，使模型完全适配目标领域。效果最好但需要大量GPU显存和训练数据。

**核心特点**：
- 所有参数都参与梯度更新
- 效果上限最高
- 显存需求最大（需要存储所有参数的梯度和优化器状态）
- 灾难性遗忘风险较高

**显存需求**（7B模型，Adam优化器）：
- 模型参数：14 GB（FP16）
- 梯度：14 GB
- 优化器状态：56 GB（FP32的参数+动量+方差）
- 总计：~84 GB

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(42)

model = nn.Sequential(
    nn.Linear(128, 256), nn.ReLU(),
    nn.Linear(256, 256), nn.ReLU(),
    nn.Linear(256, 128)
)

n_params = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)

optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4)

data = torch.randn(32, 128)
targets = torch.randn(32, 128)

print('=== Full Fine-Tuning ===')
print(f'Total parameters: {n_params:,}')
print(f'Trainable parameters: {trainable:,} ({trainable/n_params:.1%})')
print(f'\nAll parameters are updated during training.')

losses = []
for step in range(20):
    loss = F.mSELoss()(model(data), targets)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    losses.append(loss.item())

print(f'\nTraining progress:')
print(f'  Step  0: loss={losses[0]:.4f}')
print(f'  Step  5: loss={losses[5]:.4f}')
print(f'  Step 10: loss={losses[10]:.4f}')
print(f'  Step 19: loss={losses[19]:.4f}')
print(f'\nKey: Full FT updates ALL parameters, achieving best task adaptation')
print(f'but requiring the most GPU memory and training data.')